# Sudoku como CSP: fuerza bruta frente a propagación

**Facsímil 2 · Inteligencia clásica** — capítulos 5 a 7
(SAT y CSP; variables, dominios y restricciones; propagación, *backtracking* y heurísticas).

Un sudoku parece un pasatiempo, pero es la puerta de entrada a una de las familias de problemas más
útiles de la informática: los **problemas de satisfacción de restricciones** (CSP). Horarios,
asignación de aulas, configuración de productos, planificación de turnos... todos son CSP por debajo.
En este cuaderno resuelves un sudoku de dos maneras —a lo bruto y con cabeza— y **cuentas el esfuerzo
de cada una**. La diferencia es brutal, y la razón de esa diferencia —*propagar* las consecuencias de
cada decisión antes de seguir— es una idea que reaparece en agentes, planificadores y verificadores.

### Qué vas a aprender
- Qué es un **CSP**: variables, dominios y restricciones, con el sudoku como ejemplo concreto.
- Cómo funciona el **backtracking** (prueba y retrocede) y por qué, a secas, es muy lento.
- Qué es la **propagación de restricciones** (*forward checking*) y la heurística **MRV** (variable
  más restringida primero), y cuánto ahorran.
- A medir, con números reales, por qué «pensar antes de probar» convierte miles de tanteos en decenas.

### Cuánto cuesta
Unos 10 minutos. CPU, solo Python estándar.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
# Solo Python estandar.
# 0 = casilla vacia. Un sudoku de dificultad media.
TABLERO = [
    [5,3,0, 0,7,0, 0,0,0],
    [6,0,0, 1,9,5, 0,0,0],
    [0,9,8, 0,0,0, 0,6,0],
    [8,0,0, 0,6,0, 0,0,3],
    [4,0,0, 8,0,3, 0,0,1],
    [7,0,0, 0,2,0, 0,0,6],
    [0,6,0, 0,0,0, 2,8,0],
    [0,0,0, 4,1,9, 0,0,5],
    [0,0,0, 0,8,0, 0,7,9],
]
def pinta(g):
    for i, fila in enumerate(g):
        if i % 3 == 0 and i: print("------+-------+------")
        print(" ".join((str(v) if v else ".") + (" |" if j in (2,5) else "")
                        for j, v in enumerate(fila)))
pinta(TABLERO)


5 3 . | . 7 . | . . .
6 . . | 1 9 5 | . . .
. 9 8 | . . . | . 6 .
------+-------+------
8 . . | . 6 . | . . 3
4 . . | 8 . 3 | . . 1
7 . . | . 2 . | . . 6
------+-------+------
. 6 . | . . . | 2 8 .
. . . | 4 1 9 | . . 5
. . . | . 8 . | . 7 9


## 1. Un sudoku es un CSP

Un **problema de satisfacción de restricciones** tiene tres ingredientes:

- **Variables:** las cosas a decidir. Aquí, las 81 casillas.
- **Dominios:** los valores posibles de cada variable. Aquí, los números del 1 al 9 (o el valor fijo,
  si la casilla viene dada).
- **Restricciones:** las reglas que limitan qué combinaciones son válidas. Aquí: en cada fila, columna
  y caja de 3×3 no se puede repetir ningún número.

Resolver el CSP es asignar un valor a cada variable de modo que **todas** las restricciones se
cumplan. Esta forma de plantear el problema es poderosísima: el *mismo* método que resuelve el sudoku
sirve para asignar horarios o configurar una máquina, solo cambiando variables, dominios y reglas.
Empezamos por codificar las restricciones en una sola función.


In [2]:
def valido(g, fila, col, v):
    if v in g[fila]: return False                       # restriccion de fila
    if v in [g[i][col] for i in range(9)]: return False # restriccion de columna
    f0, c0 = 3*(fila//3), 3*(col//3)
    for i in range(f0, f0+3):
        for j in range(c0, c0+3):
            if g[i][j] == v: return False               # restriccion de caja
    return True
print("Las tres restricciones del sudoku, en una funcion. Todo lo demas se construye encima.")


Las tres restricciones del sudoku, en una funcion. Todo lo demas se construye encima.


## 2. Forma 1: backtracking a lo bruto

La estrategia ingenua, pero que **siempre funciona**: ve a la primera casilla vacía, prueba 1, 2, 3...
Si un valor es válido (no rompe ninguna restricción *ahora mismo*), colócalo y sigue con la siguiente
casilla. Si te atascas (ningún valor vale), **retrocede** (*backtrack*) y cambia la decisión anterior.
Es una búsqueda en profundidad sobre el espacio de asignaciones.

El problema: prueba un valor pensando solo en el presente, sin anticipar el desastre que provoca tres
casillas más adelante. Por eso explora muchísimo. Contamos cuántas veces coloca un número.


In [3]:
def resolver_bruto(g, limite=None):
    intentos = [0]
    def backtrack(g):
        if limite is not None and intentos[0] >= limite:
            return False        # nos rendimos: este metodo no escala en los dificiles
        for f in range(9):
            for c in range(9):
                if g[f][c] == 0:
                    for v in range(1, 10):
                        if valido(g, f, c, v):
                            g[f][c] = v; intentos[0] += 1
                            if backtrack(g): return True
                            g[f][c] = 0          # deshacer y probar otro
                    return False                 # ningun valor sirve -> backtrack
        return True
    sol = [fila[:] for fila in g]
    backtrack(sol)
    return sol, intentos[0]

sol1, intentos_bruto = resolver_bruto(TABLERO)
print(f"Backtracking a lo bruto: {intentos_bruto} valores colocados hasta resolverlo.")


Backtracking a lo bruto: 4208 valores colocados hasta resolverlo.


## 3. Forma 2: propagar antes de probar

Ahora con cabeza. La idea es **mirar hacia delante**. Cada casilla mantiene su **dominio**: qué valores
le quedan posibles. Y aplicamos dos mejoras clásicas de los CSP:

- ***Forward checking*** (propagación): al colocar un número, lo **tachamos** de los dominios de sus
  vecinas (su fila, su columna, su caja). Si una vecina se queda sin opciones, sabemos *de inmediato*
  que esta rama es un callejón, sin tener que llegar hasta ella.
- **Heurística MRV** (*minimum remaining values*): en vez de ir en orden, elegimos siempre la casilla
  con **menos opciones**. Es donde antes se descubre una contradicción, así que poda el árbol cuanto
  antes. Atacar primero lo más restringido es una de las heurísticas más rentables que existen.

Menos adivinar, más deducir.


In [4]:
def resolver_propagando(g):
    intentos = [0]
    dom = {}
    for f in range(9):
        for c in range(9):
            dom[(f,c)] = {g[f][c]} if g[f][c] else {v for v in range(1,10) if valido(g,f,c,v)}
    sol = [fila[:] for fila in g]

    def quita(dom, f, c, v):
        cambios = []
        vecinas = set((f,j) for j in range(9)) | set((i,c) for i in range(9))
        f0, c0 = 3*(f//3), 3*(c//3)
        vecinas |= set((i,j) for i in range(f0,f0+3) for j in range(c0,c0+3))
        for (i,j) in vecinas:
            if (i,j) != (f,c) and sol[i][j] == 0 and v in dom[(i,j)]:
                dom[(i,j)] = dom[(i,j)] - {v}; cambios.append((i,j))
        return cambios

    def backtrack():
        vacias = [(f,c) for f in range(9) for c in range(9) if sol[f][c] == 0]
        if not vacias: return True
        f, c = min(vacias, key=lambda p: len(dom[p]))   # MRV: la mas restringida primero
        for v in sorted(dom[(f,c)]):
            if valido(sol, f, c, v):
                sol[f][c] = v; intentos[0] += 1
                guardado = dom[(f,c)]; dom[(f,c)] = {v}
                cambios = quita(dom, f, c, v)           # forward checking
                if all(len(dom[p]) > 0 for p in vacias if sol[p[0]][p[1]] == 0):
                    if backtrack(): return True
                for (i,j) in cambios: dom[(i,j)].add(v) # deshacer
                dom[(f,c)] = guardado; sol[f][c] = 0
        return False

    backtrack()
    return sol, intentos[0]

sol2, intentos_prop = resolver_propagando(TABLERO)
print(f"Con propagacion + MRV: {intentos_prop} valores colocados.\n")
pinta(sol2)


Con propagacion + MRV: 51 valores colocados.

5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
------+-------+------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
------+-------+------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9


## 4. El veredicto

Mismo problema, misma respuesta, dos formas de llegar. Compara el esfuerzo:


In [5]:
print(f"A lo bruto:        {intentos_bruto:>5} colocaciones")
print(f"Propagando + MRV:  {intentos_prop:>5} colocaciones")
ahorro = 100 * (1 - intentos_prop / intentos_bruto)
print(f"\nPropagar te ahorra el {ahorro:.0f}% del trabajo. Misma solucion, una fraccion del esfuerzo.")
print("No es que el ordenador sea mas rapido: es que piensa antes de probar.")


A lo bruto:         4208 colocaciones
Propagando + MRV:     51 colocaciones

Propagar te ahorra el 99% del trabajo. Misma solucion, una fraccion del esfuerzo.
No es que el ordenador sea mas rapido: es que piensa antes de probar.


**Por qué la diferencia es tan grande.** El backtracking bruto descubre los conflictos *tarde*,
cuando ya ha construido medio tablero sobre una base imposible, y tiene que deshacerlo todo. La
propagación los descubre *temprano*: en cuanto una casilla se queda sin opciones, poda esa rama
entera. Y la heurística MRV ataca justo por donde el problema está más apretado, donde antes salta la
contradicción. Combinadas, convierten una búsqueda enorme en una manejable.


## 5. Experimento: cuanto más difícil, más se nota

La ventaja de propagar no es constante: **crece** con la dificultad del problema. Cuantas menos pistas
tiene el sudoku, más caminos sin salida hay, y más sufre el método bruto (que los recorre todos)
frente al propagado (que los poda). Lo medimos con un sudoku conocido por ser duro para la fuerza
bruta: empieza con muchas casillas vacías en la esquina.


In [6]:
DIFICIL = [
    [0,0,0, 0,0,0, 0,0,0],
    [0,0,0, 0,0,3, 0,8,5],
    [0,0,1, 0,2,0, 0,0,0],
    [0,0,0, 5,0,7, 0,0,0],
    [0,0,4, 0,0,0, 1,0,0],
    [0,9,0, 0,0,0, 0,0,0],
    [5,0,0, 0,0,0, 0,7,3],
    [0,0,2, 0,1,0, 0,0,0],
    [0,0,0, 0,4,0, 0,0,9],
]
LIMITE = 1_000_000   # le damos al metodo bruto un presupuesto de intentos
_, bruto_d = resolver_bruto(DIFICIL, limite=LIMITE)
_, prop_d  = resolver_propagando(DIFICIL)
print(f"Sudoku medio   -> bruto: {intentos_bruto:>9,} colocaciones | propagando: {intentos_prop:>6,}")
if bruto_d >= LIMITE:
    print(f"Sudoku DIFICIL -> bruto: se rinde tras {LIMITE:>9,} SIN resolverlo | propagando: {prop_d:>6,} (RESUELTO)")
    print(f"\nEn los sudokus duros, el metodo bruto ni siquiera termina dentro de un limite razonable.")
    print("Propagar sigue siendo viable: la ventaja no es lineal, se dispara con la dificultad.")
else:
    print(f"Sudoku DIFICIL -> bruto: {bruto_d:>9,} | propagando: {prop_d:>6,} (factor x{bruto_d//max(prop_d,1)})")


Sudoku medio   -> bruto:     4,208 colocaciones | propagando:     51
Sudoku DIFICIL -> bruto: se rinde tras 1,000,000 SIN resolverlo | propagando: 45,267 (RESUELTO)

En los sudokus duros, el metodo bruto ni siquiera termina dentro de un limite razonable.
Propagar sigue siendo viable: la ventaja no es lineal, se dispara con la dificultad.


## 6. Pruébalo tú

1. **Quita la heurística MRV**: recorre las casillas vacías en orden (`vacias[0]` en vez del `min`).
   ¿Suben mucho las colocaciones? Elegir *bien* la siguiente variable importa tanto como propagar.
2. **Quita el forward checking** (no llames a `quita`): vuelves casi al backtracking bruto. Mide la
   diferencia.
3. **Un sudoku sin solución** (mete un 5 repetido en la primera fila): mira cómo el método con dominios
   lo detecta antes, porque algún dominio se queda vacío enseguida.
4. **Modela otro CSP:** colorear un mapa con 3 colores sin que dos países vecinos compartan color es el
   mismo esqueleto (variables = países, dominio = colores, restricción = vecinos distintos).


## 7. Errores comunes

- **Pensar que el backtracking es inútil.** Es la base de todo; lo que lo hace práctico es añadirle
  propagación y heurísticas. Solo, escala fatal.
- **Propagar pero olvidar deshacer.** Al retroceder hay que **restaurar** los dominios que tachaste, o
  el estado queda corrupto. Es el error de implementación más común en CSP.
- **Elegir mal la variable.** Sin MRV, exploras muchísimo más. La heurística de «la más restringida
  primero» es casi gratis y muy potente.
- **Creer que esto es solo para sudokus.** El mismo método resuelve horarios, asignaciones y
  configuraciones reales. El sudoku es el *hola mundo* de los CSP.


## 8. Qué te llevas

- Un CSP son **variables, dominios y restricciones**. Modelar un problema así te da gratis un método
  general que sirve para muchísimos problemas reales.
- **Propagar** las consecuencias de cada decisión (*forward checking*) y elegir primero **la variable
  más restringida** (MRV) convierte una búsqueda gigantesca en una manejable.
- La ventaja **crece con la dificultad**: cuanto más apretado el problema, más rentable es pensar antes
  de probar.
- La lección de fondo del facsímil: no basta con encontrar *una* solución tanteando; conviene razonar
  sobre las restricciones *antes* de actuar.

**Para seguir:** el capítulo 8 usa estas restricciones como *guardrails* de un sistema; los capítulos
9-10 saltan a planificación (PDDL), que es buscar en un espacio de estados con acciones; y la idea de
«propagar y podar» reaparece en casi todo lo que decide con reglas.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*